<a href="https://colab.research.google.com/github/logankim0913/EE_467_Final_Project/blob/main/phase1and2_allbenigntraffic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import pandas as pd
import os

file_path = os.path.join("DictionaryBruteForce.pcap.csv")
#read the csv files and print the first 10 rows of each
bruteforcedata = pd.read_csv(file_path)
bruteforcedata.head(10)

spoofdata = pd.read_csv("DNS_Spoofing.pcap.csv")
spoofdata.head(10)

benigndata = pd.read_csv("BenignTraffic.pcap.csv")
benigndata1 = pd.read_csv("BenignTraffic1.pcap.csv")
benigndata2 = pd.read_csv("BenignTraffic2.pcap.csv")
benigndata3 = pd.read_csv("BenignTraffic3.pcap.csv")

benigndata.head(10)
benigndata1.head(10)
benigndata2.head(10)
benigndata3.head(10)

# #Preview statistics of the data
print("Brute Force Data Statistics:")
print(bruteforcedata.describe())
print("\nDNS Spoofing Data Statistics:")
print(spoofdata.describe())
print("\nBenign Traffic Data Statistics:")
print(benigndata.describe())
print("\nBenign Traffic Data 1")
print(benigndata1.describe())
print("\nBenign Traffic Data 2")
print(benigndata2.describe())
print("\nBenign Traffic Data 3")
print(benigndata3.describe())

#use separate benign datasets for group-aware split
train_benign = pd.concat([benigndata, benigndata1])
val_benign = benigndata2
test_benign = benigndata3

Brute Force Data Statistics:
       Header_Length  Protocol Type  Time_To_Live           Rate  \
count   13064.000000   13064.000000  13064.000000   13064.000000   
mean       23.883762       7.632961     93.120070    1732.280240   
std         7.323148       4.064012     34.447147   24243.212110   
min         0.800000       0.000000     15.600000       0.000040   
25%        18.800000       6.000000     64.000000      52.411901   
50%        24.800000       6.000000     83.100000     106.396493   
75%        30.400000       6.000000    112.500000     249.954800   
max        44.800000      17.000000    247.000000  762600.727273   

       fin_flag_number  syn_flag_number  rst_flag_number  psh_flag_number  \
count     13064.000000     13064.000000     13064.000000     13064.000000   
mean          0.026607         0.046249         0.010257         0.251944   
std           0.075118         0.089139         0.036774         0.195413   
min           0.000000         0.000000         0.

In [28]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# handle infinite values before scaling
# drop rows with infinity
train_benign_cleaned = train_benign.replace([np.inf, -np.inf], np.nan).dropna()
val_benign_cleaned = val_benign.replace([np.inf, -np.inf], np.nan).dropna()
test_benign_cleaned = test_benign.replace([np.inf, -np.inf], np.nan).dropna()

spoofdata_cleaned = spoofdata.replace([np.inf, -np.inf], np.nan).dropna()

#obtaining features as a list for each chosen class
bfd_scaled = bruteforcedata.copy()
categorical_col_bfd = []
for col in bfd_scaled.columns:
  categorical_col_bfd.append(col)
print(categorical_col_bfd)

train_benign_scaled = train_benign_cleaned.copy() #no inf
categorical_col_train_benign = []
for col in train_benign_scaled.columns:
  categorical_col_train_benign.append(col)
val_benign_scaled = val_benign_cleaned.copy() #no inf
categorical_col_val_benign = []
for col in val_benign_scaled.columns:
  categorical_col_val_benign.append(col)
test_benign_scaled = test_benign_cleaned.copy() #no inf
categorical_col_test_benign = []
for col in test_benign_scaled.columns:
  categorical_col_test_benign.append(col)

spoof_scaled = spoofdata_cleaned.copy() #no inf
categorical_col_spoof = []
for col in spoof_scaled.columns:
  categorical_col_spoof.append(col)

#Standard scaling (no outlier removal yet)
bfd_feat_std = pd.DataFrame(StandardScaler().fit_transform(bruteforcedata[categorical_col_bfd]), columns=categorical_col_bfd)
train_benign_feat_std = pd.DataFrame(StandardScaler().fit_transform(train_benign_cleaned[categorical_col_train_benign]), columns=categorical_col_train_benign)
val_benign_feat_std = pd.DataFrame(StandardScaler().fit_transform(val_benign_cleaned[categorical_col_val_benign]), columns=categorical_col_val_benign)
test_benign_feat_std = pd.DataFrame(StandardScaler().fit_transform(test_benign_cleaned[categorical_col_test_benign]), columns=categorical_col_test_benign)

spoof_feat_std = pd.DataFrame(StandardScaler().fit_transform(spoofdata_cleaned[categorical_col_spoof]), columns=categorical_col_spoof)

# Outlier removal logic
remove_outliers = True
outlier_threshold = 5

#Dictionary Brute Force Data
if remove_outliers:
    outlier_indices_bfd = set()
    for col_name in bfd_feat_std.columns:
        #Obtain column data as 2D NumPy array
        col_data = bfd_feat_std[col_name].values.astype(float)
        #Get indices of records with outlier column values
        out_indices = np.where(np.abs(col_data) > outlier_threshold)[0]
        #Merge these indices with `outlier_indices_bfd`
        outlier_indices_bfd.update(out_indices.tolist())
    #Remove outliers
    bfd_feat_std = bfd_feat_std.drop(index=list(outlier_indices_bfd))

    # Benign Data Train
    outlier_indices_train_benign = set()
    for col_name in train_benign_feat_std.columns:
        #Obtain column data as 2D NumPy array
        col_data = train_benign_feat_std[col_name].values.astype(float)
        #Get indices of records with outlier column values
        out_indices = np.where(np.abs(col_data) > outlier_threshold)[0]
        # Merge these indices with `outlier_indices_benign`
        outlier_indices_train_benign.update(out_indices.tolist())
    #Remove outliers
    train_benign_feat_std = train_benign_feat_std.drop(index=list(outlier_indices_train_benign))

    # Benign Data Val
    outlier_indices_val_benign = set()
    for col_name in val_benign_feat_std.columns:
        #Obtain column data as 2D NumPy array
        col_data = val_benign_feat_std[col_name].values.astype(float)
        #Get indices of records with outlier column values
        out_indices = np.where(np.abs(col_data) > outlier_threshold)[0]
        # Merge these indices with `outlier_indices_benign`
        outlier_indices_val_benign.update(out_indices.tolist())
    #Remove outliers
    val_benign_feat_std = val_benign_feat_std.drop(index=list(outlier_indices_val_benign))

    # Benign Data Test
    outlier_indices_test_benign = set()
    for col_name in test_benign_feat_std.columns:
        #Obtain column data as 2D NumPy array
        col_data = test_benign_feat_std[col_name].values.astype(float)
        #Get indices of records with outlier column values
        out_indices = np.where(np.abs(col_data) > outlier_threshold)[0]
        # Merge these indices with `outlier_indices_benign`
        outlier_indices_test_benign.update(out_indices.tolist())
    #Remove outliers
    test_benign_feat_std = test_benign_feat_std.drop(index=list(outlier_indices_test_benign))

    # DNS Spoofing Data
    outlier_indices_spoof = set()
    for col_name in spoof_feat_std.columns:
        #Obtain column data as 2D NumPy array
        col_data = spoof_feat_std[col_name].values.astype(float)
        #Get indices of records with outlier column values
        out_indices = np.where(np.abs(col_data) > outlier_threshold)[0]
        #Merge these indives with 'outlier_indices_spoof"
        outlier_indices_spoof.update(out_indices.tolist())
    #Remove outliers
    spoof_feat_std = spoof_feat_std.drop(index=list(outlier_indices_spoof))

print("Brute Force Scaled Data (without outliers):")
print(bfd_feat_std.head())
print("\nTrain Benign Scaled Data (without outliers):")
print(train_benign_feat_std.head())
print("\nVal Benign Scaled Data (without outliers):")
print(val_benign_feat_std.head())
print("\nTest Benign Scaled Data (without outliers):")
print(test_benign_feat_std.head())
print("\nSpoof Scaled Data (without outliers):")
print(spoof_feat_std.head())

['Header_Length', 'Protocol Type', 'Time_To_Live', 'Rate', 'fin_flag_number', 'syn_flag_number', 'rst_flag_number', 'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number', 'ack_count', 'syn_count', 'fin_count', 'rst_count', 'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC', 'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP', 'IGMP', 'IPv', 'LLC', 'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number', 'Variance']
Brute Force Scaled Data (without outliers):
   Header_Length  Protocol Type  Time_To_Live      Rate  fin_flag_number  \
0       0.452861      -0.401825      0.701970 -0.070212         2.308350   
1       0.835224      -0.401825      0.150379 -0.070211        -0.354221   
2      -0.584984      -0.401825     -0.232831 -0.066449        -0.354221   
3       0.288990      -0.401825      0.710679 -0.070138        -0.354221   
4      -0.967348       2.304963      1.070665 -0.067000        -0.354221   

   syn_flag_number  rst_flag_number  psh_flag_number  ack_fl

In [29]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def clean_df(df):
    #Replaces inf/-inf with NaN and drops those selected rows
    return df.replace([np.inf, -np.inf], np.nan).dropna()

skew_cols = ["Rate", "Tot sum", "Tot size", "IAT", "Variance", "Max", "AVG", "Std", "Min", "Header_Length"]

def safe_log1p(df, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns:
            # ensure numeric
            df[c] = pd.to_numeric(df[c], errors="coerce")
            # clip negatives to 0 just in case (these should be >=0 anyway)
            df[c] = df[c].clip(lower=0)
            df[c] = np.log1p(df[c])
    return df
#scaling only on benign data
bfd = pd.read_csv("DictionaryBruteForce.pcap.csv")
spoof = pd.read_csv("DNS_Spoofing.pcap.csv")
benign = pd.read_csv("BenignTraffic.pcap.csv")
benign1 = pd.read_csv("BenignTraffic1.pcap.csv")
benign2 = pd.read_csv("BenignTraffic2.pcap.csv")
benign3 = pd.read_csv("BenignTraffic3.pcap.csv")
#use different datasets for train/val/test benign
train_benign = pd.concat([benign, benign1])
val_benign = benign2
test_benign = benign3


#all features extract
bfd_copy = bfd.copy()
categorical_col = []
for col in bfd_copy.columns:
  categorical_col.append(col)

#clean data so no inf
bfd = clean_df(bfd)
spoof = clean_df(spoof)
benign_train = clean_df(train_benign)
benign_val = clean_df(val_benign)
benign_test = clean_df(test_benign)

#Test split
RANDOM_SEED = 42

#benign data split so train = 70%, val and test both 15%
# benign_train, benign_tmp = train_test_split(
#     benign, test_size=0.30, random_state=RANDOM_SEED, shuffle=True
# )
# benign_val, benign_test = train_test_split(
#     benign_tmp, test_size=0.50, random_state=RANDOM_SEED, shuffle=True
# )

syn_data_columns = benign_train.columns # Assuming benign_train has all the relevant columns

scaler = StandardScaler()
#training set for benign
X_train = scaler.fit_transform(benign_train[syn_data_columns].values)
#validation set for benign
X_val_benign = scaler.transform(benign_val[syn_data_columns].values)

# test set includes benign_test + all attacks
test = pd.concat(
    [
        benign_test.assign(label=0),   # 0 = benign
        bfd.assign(label=1),           # 1 = attack
        spoof.assign(label=1),
    ],
    ignore_index=True
)

#test set has both benign and attack
X_test = scaler.transform(test[categorical_col].values)
#labels for test set
y_test = test["label"].values

print("Shapes:")
print("X_train (benign):", X_train.shape)
print("X_val_benign:", X_val_benign.shape)
print("X_test (mixed):", X_test.shape, "y_test:", y_test.shape)

print("\nTest class balance:")
print(pd.Series(y_test).value_counts())

#my output was 191957 for attack (1) and 54352 for benign (0)
#HIGHLY IMBALANCED (much more attack)

Shapes:
X_train (benign): (657907, 39)
X_val_benign: (310395, 39)
X_test (mixed): (321781, 39) y_test: (321781,)

Test class balance:
1    191957
0    129824
Name: count, dtype: int64


In [30]:
bfd_train, bfd_test = train_test_split(
    bfd, test_size=0.30, random_state=RANDOM_SEED, shuffle=True
)

spoof_train, spoof_test = train_test_split(
    spoof, test_size=0.30, random_state=RANDOM_SEED, shuffle=True
)


train_full = pd.concat(
    [
        benign_train.assign(label=0),
        bfd_train.assign(label=1),
        spoof_train.assign(label=1),
    ]
)

test = pd.concat(
    [
        benign_test.assign(label=0),
        bfd_test.assign(label=1),
        spoof_test.assign(label=1),
    ],
    ignore_index=True
)


X_train_full = scaler.transform(train_full[categorical_col].values)
y_train_full = train_full["label"].values


#imports
import numpy as np
import matplotlib.pyplot as plt

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

In [31]:
#Built autoencoder
def build_autoencoder(input_dim):

    inputs = keras.Input(shape=(input_dim,))

    # Encoder
    x = layers.Dense(32, activation='relu')(inputs)
    x = layers.Dense(16, activation='relu')(x)
    latent = layers.Dense(8, activation='relu')(x)

    # Decoder
    x = layers.Dense(16, activation='relu')(latent)
    x = layers.Dense(32, activation='relu')(x)
    outputs = layers.Dense(input_dim)(x)

    model = keras.Model(inputs, outputs)

    encoder = keras.Model(inputs, latent)

    model.compile(
        optimizer='adam',
        loss='mse'
    )

    return model, encoder

In [32]:
#instantiate model
ae, encoder = build_autoencoder(X_train.shape[1])

#parameter count
def count_parameters(model):
    return np.sum([np.prod(v.shape) for v in model.trainable_weights])

print("Trainable Parameters:", count_parameters(ae))

Trainable Parameters: 3919


In [34]:
#Train autoencoder
import numpy as np

def add_gaussian_noise(X, sigma=0.05):
    Xn = X + np.random.normal(0.0, sigma, size=X.shape)
    return Xn

sigma = 0.05  # try 0.02, 0.05, 0.1
X_train_noisy = add_gaussian_noise(X_train, sigma=sigma)

history = ae.fit(
    X_train_noisy, X_train,          # noisy input, clean target
    validation_data=(X_val_benign, X_val_benign),
    epochs=40,
    batch_size=512,
    verbose=1
)

# history = ae.fit(
#     X_train,
#     X_train,
#     epochs=40,
#     batch_size=256,
#     validation_data=(X_val_benign, X_val_benign),
#     verbose=1
# )

Epoch 1/40
1285/1285 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - loss: 0.0368 - val_loss: 0.0290
Epoch 2/40
1285/1285 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0348 - val_loss: 0.0280
Epoch 3/40
1285/1285 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 0.0334 - val_loss: 0.0264
Epoch 4/40
1285/1285 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0321 - val_loss: 0.0270
Epoch 5/40
1285/1285 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 0.0314 - val_loss: 0.0256
Epoch 6/40
1285/1285 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - loss: 0.0310 - val_loss: 0.0245
Epoch 7/40
1285/1285 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - loss: 0.0301 - val_loss: 0.0250
Epoch 8/40
1285/1285 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 0.0293 - val_loss: 0.0240
Epoch 9/40
1285/1285 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.0289 - val_loss: 0.0234
Epoch 10/40
1285/1285 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 0.0281 - val_loss: 0.0233
Epoch 11/40
1285/1285 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 0.0282 - val_loss: 0.0232
Epoch 12/40
1285/1285 ━━━━━━

In [35]:
#Reconstruction error
def reconstruction_error(model, X):
    reconstructed = model.predict(X, verbose=0)
    error = np.mean((X - reconstructed) ** 2, axis=1)
    return error

In [36]:
#Threshold at 0.5% FPR
def compute_threshold(val_errors, target_fpr=0.005):
    return np.percentile(val_errors, 100 * (1 - target_fpr))

val_errors = reconstruction_error(ae, X_val_benign)
threshold = compute_threshold(val_errors, 0.005)

print("0.5% FPR Threshold:", threshold)

0.5% FPR Threshold: 0.2799756945912267


In [37]:
#AE detection on test set
def ae_predict(model, X, threshold):
    errors = reconstruction_error(model, X)
    preds = (errors > threshold).astype(int)
    return preds, errors

ae_preds, ae_errors = ae_predict(ae, X_test, threshold)

In [38]:
def count_lr_params(model):
  # Coefficients (one per feature per class, but binary only needs one set)
  n_coefficients = logreg.coef_.size

  # Intercepts (one per class, binary only needs one)
  n_intercepts = logreg.intercept_.size

  total_params = n_coefficients + n_intercepts
  print(f"Total parameters: {total_params}")


In [39]:
#Logistic Regression baseline
logreg = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    n_jobs=-1
)

logreg.fit(X_train_full, y_train_full)
lr_preds = logreg.predict(X_test)
count_lr_params(logreg)

Total parameters: 40


In [40]:
#Autoencoder Evaluation
from sklearn.metrics import classification_report
tn, fp, fn, tp = confusion_matrix(y_test, ae_preds).ravel()

ae_recall = tp / (tp + fn)
ae_fpr = fp / (fp + tn)

#Logistic Regression Evaluation
tn, fp, fn, tp = confusion_matrix(y_test, lr_preds).ravel()

lr_recall = tp / (tp + fn)
lr_fpr = fp / (fp + tn)

print("AE Recall @ 0.5% FPR:", ae_recall)
print("AE Actual FPR:", ae_fpr, "\n")

print("AE Classification Report: \n\n", classification_report(y_test, ae_preds))
print("\nLogistic Regression Classification Report: \n\n", classification_report(y_test, lr_preds))

print("LogReg Recall:", lr_recall)
print("LogReg FPR:", lr_fpr, "\n")

AE Recall @ 0.5% FPR: 0.05613236297712509
AE Actual FPR: 0.00642408183386739 

AE Classification Report: 

               precision    recall  f1-score   support

           0       0.42      0.99      0.59    129824
           1       0.93      0.06      0.11    191957

    accuracy                           0.43    321781
   macro avg       0.67      0.52      0.35    321781
weighted avg       0.72      0.43      0.30    321781


Logistic Regression Classification Report: 

               precision    recall  f1-score   support

           0       0.70      0.79      0.74    129824
           1       0.84      0.77      0.80    191957

    accuracy                           0.78    321781
   macro avg       0.77      0.78      0.77    321781
weighted avg       0.78      0.78      0.78    321781

LogReg Recall: 0.7671613955208718
LogReg FPR: 0.21329646290362336 



In [41]:
def distance_scoring(encoder, kmeans, X):
  latent = encoder.predict(X, verbose=0)
  distances = kmeans.transform(latent)
  min_distances = distances.min(axis=1)
  return min_distances

def ae_kmeans_predict(encoder, kmeans, X, threshold):
    distances = distance_scoring(encoder, kmeans, X)
    preds = (distances > threshold).astype(int)
    return preds, distances

In [42]:
from sklearn.cluster import MiniBatchKMeans, KMeans
latent_train = encoder.predict(X_train)
kmeans = KMeans(random_state=0, max_iter=500)
kmeans.fit(latent_train)

distances = distance_scoring(encoder, kmeans, X_val_benign)
scoring_threshold = np.percentile(distances, 99.5)

ae_km_preds, ae_km_distances = ae_kmeans_predict(encoder, kmeans, X_test, scoring_threshold)

20560/20560 ━━━━━━━━━━━━━━━━━━━━ 24s 1ms/step


In [43]:
tn, fp, fn, tp = confusion_matrix(y_test, ae_km_preds).ravel()

ae_k_recall = tp / (tp + fn)
ae_k_fpr = fp / (fp + tn)

print("AE K-means Recall @ 0.5% FPR:", ae_k_recall)
print("AE K-Means Actual FPR:", ae_k_fpr, "\n")

print("AE Classification Report: \n\n", classification_report(y_test, ae_km_preds))

AE K-means Recall @ 0.5% FPR: 0.030418270758555303
AE K-Means Actual FPR: 0.008049359132363815 

AE Classification Report: 

               precision    recall  f1-score   support

           0       0.41      0.99      0.58    129824
           1       0.85      0.03      0.06    191957

    accuracy                           0.42    321781
   macro avg       0.63      0.51      0.32    321781
weighted avg       0.67      0.42      0.27    321781



In [44]:
def count_rf_params(model):
    total_nodes = sum(tree.tree_.node_count for tree in model.estimators_)

    print(f"Number of trees: {len(model.estimators_)}")
    print(f"Total nodes: {total_nodes}")

In [45]:
from sklearn.ensemble import RandomForestClassifier
N_ENSEMBLE_CPUS = max(os.cpu_count()//2, 1)
rf_15_model = RandomForestClassifier(n_estimators=15, n_jobs=N_ENSEMBLE_CPUS).fit(X_train_full, y_train_full)
rf_15_preds = rf_15_model.predict(X_test)

tn, fp, fn, tp = confusion_matrix(y_test, rf_15_preds).ravel()

rf_recall = tp / (tp + fn)
rf_fpr = fp / (fp + tn)

print("\nRF Classification Report: \n\n", classification_report(y_test, rf_15_preds))

print("RF 15 Recall:", rf_recall)
print("RF 15 FPR:", rf_fpr, "\n")

print("Random Forest Classifier Model Features:")
count_rf_params(rf_15_model)



RF Classification Report: 

               precision    recall  f1-score   support

           0       0.91      0.99      0.95    129824
           1       1.00      0.94      0.97    191957

    accuracy                           0.96    321781
   macro avg       0.95      0.97      0.96    321781
weighted avg       0.96      0.96      0.96    321781

RF 15 Recall: 0.937189057966107
RF 15 FPR: 0.006539622874044861 

Random Forest Classifier Model Features:
Number of trees: 15
Total nodes: 921269
